### Try to understand the dataset 

In [4]:
import json
import sys
from pathlib import Path
from openai import OpenAI, APIError

In [5]:
data_path = Path("../../data/convfinqa_dataset.json")
data = json.load(open(data_path))

In [6]:
# basic information
print("Keys in the dataset:", data.keys())

Keys in the dataset: dict_keys(['train', 'dev'])


In [7]:
train_data = data.get("train", [])
dev_data = data.get("dev", [])
all_data = train_data + dev_data

print("Number of train samples:", len(train_data))
print("Number of dev samples:", len(dev_data))
print("Total number of samples:", len(all_data))

Number of train samples: 3037
Number of dev samples: 421
Total number of samples: 3458


In [8]:
train_type2 = sum(1 for sample in train_data if sample["features"]["has_type2_question"])
print("Number of Type 2 training samples:", train_type2)

dev_type2 = sum(1 for sample in dev_data if sample["features"]["has_type2_question"])
print("Number of Type 2 development samples:", dev_type2)

Number of Type 2 training samples: 889
Number of Type 2 development samples: 121


In [35]:
# find a sample with type 2 question
sample_with_type2 = next((sample for sample in all_data if sample["features"]["has_type2_question"]), None)
if sample_with_type2:
    print("Sample with Type 2 question found:")
    print(json.dumps(sample_with_type2, indent=2))

Sample with Type 2 question found:
{
  "id": "Double_UPS/2009/page_33.pdf",
  "doc": {
    "pre_text": "( 1 ) includes shares repurchased through our publicly announced share repurchase program and shares tendered to pay the exercise price and tax withholding on employee stock options . shareowner return performance graph the following performance graph and related information shall not be deemed 201csoliciting material 201d or to be 201cfiled 201d with the securities and exchange commission , nor shall such information be incorporated by reference into any future filing under the securities act of 1933 or securities exchange act of 1934 , each as amended , except to the extent that the company specifically incorporates such information by reference into such filing . the following graph shows a five-year comparison of cumulative total shareowners 2019 returns for our class b common stock , the s&p 500 index , and the dow jones transportation average . the comparison of the total cumul

In [10]:
# check whether every sample has a non-empty table
empty_table_train = []
non_empty_table_train = []

for idx, record in enumerate(train_data):
    table = record.get("doc", {}).get("table", [])
    if not table:
        empty_table_train.append(idx)
    else:
        non_empty_table_train.append(idx)

print(len(empty_table_train))
print(len(non_empty_table_train))

empty_table_dev = []
non_empty_table_dev = []

for idx, record in enumerate(dev_data):
    table = record.get("doc", {}).get("table", [])
    if not table:
        empty_table_dev.append(idx)
    else:
        non_empty_table_dev.append(idx)

print(len(empty_table_dev))
print(len(non_empty_table_dev))

0
3037
0
421


In [ ]:
# statistics on different operations and data selections in the dataset
from collections import Counter
import re

all_operations = Counter()
for record in all_data:
    programs = record["dialogue"]["turn_program"]
    for program in programs:
        operation_list = re.findall(r"(\w+)\(", program)
        all_operations.update(operation_list)

for operation, count in all_operations.items():
    print(f"Operation: {operation}, Count: {count}")


# check number of single number selection programs
single_number_selection_count = 0
for record in all_data:
    programs = record["dialogue"]["turn_program"]
    for program in programs:
        if program and not any(op in programs for op in ["filter", "sort", "aggregate", "arithmetic", "comparison", "join"]):
            single_number_selection_count += 1

print(f"Single number selection programs: {single_number_selection_count}")

Operation: subtract, Count: 5131
Operation: divide, Count: 4280
Operation: add, Count: 2457
Operation: multiply, Count: 894
Operation: greater, Count: 40
Operation: exp, Count: 4
Single number selection programs: 12594


In [18]:
# the paper mentionens:
# "For the number selection questions depending on previous references, e.g., 'what is that value in the
#  subsequent year?', the model is mostly able to answer. Also, the model is mostly clear on when to discard 
# the previous context and make the transition to new questions."

# try to find some keywords that indicate the model needs to refer to previous context

previous_context_keywords = ["that", "subsequent", "this", "previous", "prior", "earlier", "former", "last", "before", "preceding", "prior"]

questions_with_previous_context = []
for record in all_data:
    for i, q in enumerate(record["dialogue"]["conv_questions"]):
        if i > 0:
            if any(keyword in q.lower() for keyword in previous_context_keywords):
                questions_with_previous_context.append(q)

print(len(questions_with_previous_context))

3221


In [22]:
from collections import Counter
import statistics

# try to get the distribution of the number of turns in the data
turn_counts = [len(rec["dialogue"]["conv_questions"]) for rec in all_data]

print(f"min: {min(turn_counts)}")
print(f"max: {max(turn_counts)}")
print(f"avg: {statistics.mean(turn_counts):.2f}")
print(f"median: {statistics.median(turn_counts)}")

turn_dist = Counter(turn_counts)
print(f"\n distribution:")
for turns in sorted(turn_dist.keys()):
    count = turn_dist[turns]
    pct = 100 * count / len(all_data)
    print(f"  {turns}: {count:4d} ({pct:5.1f}%)")

print("\ntry to estimation the tokens needed")
max_tokens_needed = 500 + (max(turn_counts) * 50)
avg_tokens_needed = 500 + (statistics.mean(turn_counts) * 50)
print(f"max context: ~{max_tokens_needed} tokens")
print(f"avg context: ~{avg_tokens_needed:.0f} tokens")

min: 1
max: 9
avg: 3.64
median: 4.0

 distribution:
  1:    6 (  0.2%)
  2:  858 ( 24.8%)
  3:  751 ( 21.7%)
  4:  940 ( 27.2%)
  5:  649 ( 18.8%)
  6:  184 (  5.3%)
  7:   52 (  1.5%)
  8:   16 (  0.5%)
  9:    2 (  0.1%)

try to estimation the tokens needed
max context: ~950 tokens
avg context: ~682 tokens


In [ ]:
# how important are reference words regarding hybrid dialogues
reference_keywords = ["that", "subsequent", "this", "previous", "prior", "earlier", "former", "last", "before", "preceding", "prior"]

hybrid_with_refs = 0
hybrid_turns_with_refs = 0
hybrid_total_turns = 0

for rec in all_data:
    qa_split = rec["dialogue"]["qa_split"]
    conv_questions = rec["dialogue"]["conv_questions"]
    
    if not any(qa_split):  # is qa_split all false, then not a hybird
        continue
    
    has_ref = False
    for turn_idx, q in enumerate(conv_questions):
        hybrid_total_turns += 1
        q_lower = q.lower()
        
        # check if any contains reference keywords
        if any(kw in q_lower for kw in reference_keywords):
            hybrid_turns_with_refs += 1
            has_ref = True
    
    if has_ref:
        hybrid_with_refs += 1

print(f"all hybrid: {sum(1 for r in all_data if any(r['dialogue']['qa_split']))}")
print(f"hybrid_with_refs: {hybrid_with_refs}")
print(f"  → {100*hybrid_with_refs/sum(1 for r in all_data if any(r['dialogue']['qa_split'])):.1f}%")
print(f"\nhybrid_total_turns {hybrid_total_turns}")
print(f"hybrid_turns_with_refs: {hybrid_turns_with_refs}")
print(f"  → {100*hybrid_turns_with_refs/hybrid_total_turns:.1f}%")

# this could be a good indicator that explicit reference handling could be beneficial

all hybrid: 1010
hybrid_with_refs: 741
  → 73.4%

hybrid_total_turns 3968
hybrid_turns_with_refs: 1521
  → 38.3%
